# Logos CPU Debug Run

Small CPU-only dry run for the current Logos architecture. It enables KDA, SWA, CSA, HCA, MoE, Block Attention Residuals, chunked LM-head CE, and router-bias updates with tiny parameters so it can run on a laptop.


In [ ]:
import torch
from scripts.train import build_arg_parser, build_model

parser = build_arg_parser()
args = parser.parse_args([
    '--model', 'logos',
    '--device', 'cpu',
    '--d-model', '32',
    '--num-heads', '4',
    '--head-dim', '8',
    '--d-ff', '64',
    '--max-length', '16',
    '--num-entry-layers', '1',
    '--num-body-layers', '2',
    '--num-exit-layers', '1',
    '--num-loops', '2',
    '--entry-attn-pattern', 'hca',
    '--body-attn-pattern', 'csa,kda,swa,hca',
    '--exit-attn-pattern', 'csa',
    '--csa-compression', '4',
    '--csa-top-k', '2',
    '--csa-indexer-heads', '2',
    '--csa-indexer-dim', '4',
    '--hca-compression', '8',
    '--swa-window', '4',
    '--chunk-size', '4',
    '--conv-size', '2',
    '--use-moe',
    '--num-shared-experts', '1',
    '--num-sparse-experts', '4',
    '--top-k', '2',
    '--expert-d-ff', '16',
    '--capacity-factor', '2.0',
    '--moe-diversity-factor', '0.5',
    '--lm-head-chunk-size', '8',
    '--gradient-checkpointing',
    '--ckpt-granularity', 'per-block',
])
cfg, model = build_model(args, vocab_size=128)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
ids = torch.randint(0, cfg.vocab_size, (2, 16), dtype=torch.long)
out = model(ids, labels=ids)
loss = out['loss']
assert loss is not None and torch.isfinite(loss), loss
loss.backward()
optimizer.step()
model.update_router_biases(out['topk_indices'])
print('CPU debug run OK')
print('loss:', float(loss.detach()))
print('entry:', model.entry_attn_schedule)
print('body:', model.body_attn_schedule)
print('exit:', model.exit_attn_schedule)
